# Cyclistic Bike Share — Pandas Prototype

Input: **`trips_prototype_with_weather.csv`** (~148k rows).  
Each step is individually timed. Final cell writes all timings to **`metrics.json`**.   
Contains two queries suited for MapReduce that will show slower timings at this scale.

In [1]:
import os
import time
import json
import platform
import shutil
from pathlib import Path

import pandas as pd
import psutil

notebook_start = time.perf_counter()
timings: dict = {}

print("===== CPU =====")
print(f"Processor : {platform.processor() or 'n/a'}")
print(f"Physical cores : {psutil.cpu_count(logical=False)}")
print(f"Logical CPUs   : {psutil.cpu_count(logical=True)}")

print("\n===== RAM =====")
vm = psutil.virtual_memory()
print(f"Total     : {vm.total / (1024**3):.2f} GB")
print(f"Available : {vm.available / (1024**3):.2f} GB")

print("\n===== Disk (cwd) =====")
usage = shutil.disk_usage(".")
print(f"Total : {usage.total / (1024**3):.2f} GB")
print(f"Free  : {usage.free  / (1024**3):.2f} GB")

print(f"\n===== Python =====")
import sys; print(sys.version)

===== CPU =====
Processor : Intel64 Family 6 Model 126 Stepping 5, GenuineIntel
Physical cores : 2
Logical CPUs   : 4

===== RAM =====
Total     : 7.77 GB
Available : 1.59 GB

===== Disk (cwd) =====
Total : 236.86 GB
Free  : 22.01 GB

===== Python =====
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


## Load & Preprocess
Read CSV, parse timestamps, derive `ride_length_minutes`, `hour`, and `day_of_week`.  
Timed as a single block — equivalent to the SQL notebook's CSV ingest + `to_sql` load step.

In [2]:
CSV_PATH = Path("trips_prototype_with_weather.csv")
if not CSV_PATH.is_file():
    alt = Path.cwd() / "Prototype" / "trips_prototype_with_weather.csv"
    if alt.is_file():
        CSV_PATH = alt
    else:
        raise FileNotFoundError(f"trips_prototype_with_weather.csv not found. Tried: {CSV_PATH.resolve()} and {alt}")

print(f"Using CSV: {CSV_PATH.resolve()}")

t0 = time.perf_counter()

df = pd.read_csv(CSV_PATH, dtype={"ride_id": str})
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"]   = pd.to_datetime(df["ended_at"],   utc=True)
df["ride_length_minutes"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
df["hour"]        = df["started_at"].dt.hour
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_of_week"] = pd.Categorical(df["started_at"].dt.day_name(), categories=day_order, ordered=True)

load_s = time.perf_counter() - t0
timings["load_preprocess"] = round(load_s, 4)

print(f"Rows loaded  : {len(df):,}")
print(f"Columns      : {list(df.columns)}")
print(f"Elapsed      : {load_s:.4f}s")
df.head(3)

Using CSV: C:\Users\imonl\BigData\Cyclistic_Data\Prototype\trips_prototype_with_weather.csv
Rows loaded  : 148,045
Columns      : ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'date', 'PRCP', 'TMAX', 'TMIN', 'ride_length_minutes', 'hour', 'day_of_week']
Elapsed      : 0.7700s


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,date,PRCP,TMAX,TMIN,ride_length_minutes,hour,day_of_week
0,53360EC85A03516C,electric_bike,2020-11-05 15:50:30+00:00,2020-11-05 16:08:25+00:00,NaN,NaN,Western Ave & Congress Pkwy,382.0,41.930000,-87.710000,41.874704,-87.686426,casual,2020-11-05,0.0,71.0,56.0,17.916667,15,Thursday
1,05E0A2BD796D7CEE,classic_bike,2022-06-26 12:11:44+00:00,2022-06-26 13:19:51+00:00,Broadway & Argyle St,13108,Sheridan Rd & Montrose Ave,TA1307000107,41.973815,-87.659660,41.961670,-87.654640,casual,2022-06-26,0.0,83.0,68.0,68.116667,12,Sunday
2,3B995854F4E5F068,classic_bike,2022-09-17 23:50:22+00:00,2022-09-18 00:01:37+00:00,Wells St & Concord Ln,TA1308000050,Clark St & Elm St,TA1307000039,41.912133,-87.634656,41.902973,-87.631280,member,2022-09-17,0.0,88.0,71.0,11.250000,23,Saturday


## Row Count & Data Quality
Full-pass scan: total rows, missing station names, missing end coordinates, invalid durations (≤ 0 minutes).  
Equivalent to the SQL `COUNT` + `SUM(CASE WHEN ...)` quality query.

In [3]:
t0 = time.perf_counter()

total_rows            = len(df)
missing_start_station = df["start_station_name"].isna().sum() + (df["start_station_name"] == "").sum()
missing_end_station   = df["end_station_name"].isna().sum()   + (df["end_station_name"]   == "").sum()
missing_end_coords    = (df["end_lat"].isna() | df["end_lng"].isna()).sum()
invalid_durations     = (df["ride_length_minutes"] <= 0).sum()

quality_s = time.perf_counter() - t0
timings["row_count_quality"] = round(quality_s, 4)

quality_df = pd.DataFrame([{
    "total_rows"           : total_rows,
    "missing_start_station": int(missing_start_station),
    "missing_end_station"  : int(missing_end_station),
    "missing_end_coords"   : int(missing_end_coords),
    "invalid_durations"    : int(invalid_durations),
}])
print(f"Elapsed: {quality_s:.4f}s")
display(quality_df)

Elapsed: 0.0540s


,total_rows,missing_start_station,missing_end_station,missing_end_coords,invalid_durations
0,148045,16102,17453,165,106


## Rides by Member Type & Bike Type
Two simple GROUP BY aggregations. Low cardinality — fast everywhere.  
Timed individually to match SQL step timings.

In [4]:
t0 = time.perf_counter()
rides_by_member = (
    df.groupby("member_casual", sort=False)
      .size()
      .reset_index(name="total_rides")
      .sort_values("total_rides", ascending=False)
)
member_s = time.perf_counter() - t0
timings["rides_by_member_type"] = round(member_s, 4)
print(f"===== Rides by member type =====\nElapsed: {member_s:.4f}s")
display(rides_by_member)

# --- 4b: Rides by bike type ---
t0 = time.perf_counter()
rides_by_bike = (
    df.groupby("rideable_type", sort=False)
      .size()
      .reset_index(name="total_rides")
      .sort_values("total_rides", ascending=False)
)
bike_s = time.perf_counter() - t0
timings["rides_by_bike_type"] = round(bike_s, 4)
print(f"\n===== Rides by bike type =====\nElapsed: {bike_s:.4f}s")
display(rides_by_bike)

===== Rides by member type =====
Elapsed: 0.0297s


,member_casual,total_rides
1,member,86000
0,casual,62045



===== Rides by bike type =====
Elapsed: 0.0231s


,rideable_type,total_rides
1,classic_bike,59391
0,electric_bike,54176
2,docked_bike,34478


## Average Ride Duration by Member Type
Aggregation (AVG, MIN, MAX) over the derived `ride_length_minutes` column, filtered to valid durations.  
Equivalent to the SQL `julianday()` arithmetic query in Step 11.

In [5]:
t0 = time.perf_counter()

valid = df[df["ride_length_minutes"] > 0]
avg_duration = (
    valid.groupby("member_casual")["ride_length_minutes"]
         .agg(avg_duration_minutes="mean", min_duration_minutes="min", max_duration_minutes="max")
         .round(2)
         .reset_index()
)

duration_s = time.perf_counter() - t0
timings["avg_duration_by_member"] = round(duration_s, 4)

print(f"===== Avg ride duration by member type =====\nElapsed: {duration_s:.4f}s")
display(avg_duration)

===== Avg ride duration by member type =====
Elapsed: 0.0635s


,member_casual,avg_duration_minutes,min_duration_minutes,max_duration_minutes
0,casual,32.32,0.02,20679.02
1,member,13.70,0.02,1499.93


## Popular Routes *(MapReduce Candidate 1)*
GROUP BY on the composite key `(start_station_name, end_station_name)`.  
At prototype scale this produces ~10k–50k unique key pairs; at 14.8M rows it exceeds 500k.  
**Why?** the key space grows quadratically with the number of stations, 
making in-memory hash aggregation the bottleneck for both pandas and SQLite. 
Spark's distributed shuffle + hash aggregation eliminates that bottleneck by partitioning keys across workers.

In [6]:
t0 = time.perf_counter()

routes = (
    df.dropna(subset=["start_station_name", "end_station_name"])
      .query("start_station_name != '' and end_station_name != ''")
      .groupby(["start_station_name", "end_station_name"], sort=False)
      .size()
      .reset_index(name="route_count")
      .sort_values("route_count", ascending=False)
)

routes_s = time.perf_counter() - t0
timings["popular_routes_mapreduce_1"] = round(routes_s, 4)

print(f"===== Popular routes (MapReduce Candidate 1) =====\nElapsed: {routes_s:.4f}s")
print(f"Unique routes found: {len(routes):,}")
display(routes.head(10))

===== Popular routes (MapReduce Candidate 1) =====
Elapsed: 0.1993s
Unique routes found: 46,755


,start_station_name,end_station_name,route_count
161,Streeter Dr & Grand Ave,Streeter Dr & Grand Ave,322
2396,Millennium Park,Millennium Park,163
8,Michigan Ave & Oak St,Michigan Ave & Oak St,161
1276,Ellis Ave & 60th St,University Ave & 57th St,136
784,Ellis Ave & 60th St,Ellis Ave & 55th St,130
1323,Indiana Ave & Roosevelt Rd,Indiana Ave & Roosevelt Rd,128
132,Ellis Ave & 55th St,Ellis Ave & 60th St,123
16,DuSable Lake Shore Dr & Monroe St,DuSable Lake Shore Dr & Monroe St,120
541,University Ave & 57th St,Ellis Ave & 60th St,119
293,Lake Shore Dr & Monroe St,Lake Shore Dr & Monroe St,116


## Busiest Stations — Combined Traffic *(MapReduce Candidate 2)*
Union of departures and arrivals per station, then GROUP BY + SUM.  
**Why?** requires two independent full-table scans (one for start, one for end), 
followed by a reduce step to merge counts per station key. 
This maps directly to the canonical MapReduce pattern: two Map phases emit `(station, 1)` events; 
the Reduce phase sums them. Spark can execute both scans in parallel on separate partitions, 
whereas SQLite and pandas must execute them sequentially.

In [7]:
t0 = time.perf_counter()

valid_start = df["start_station_name"].dropna()
valid_start = valid_start[valid_start != ""]

valid_end = df["end_station_name"].dropna()
valid_end = valid_end[valid_end != ""]

departures = valid_start.value_counts().rename("departures")
arrivals   = valid_end.value_counts().rename("arrivals")

station_activity = (
    pd.concat([departures, arrivals], axis=1)
      .fillna(0)
      .astype(int)
)
station_activity["total_activity"] = station_activity["departures"] + station_activity["arrivals"]
station_activity = station_activity.sort_values("total_activity", ascending=False).reset_index()
station_activity.rename(columns={"index": "station_name"}, inplace=True)

stations_s = time.perf_counter() - t0
timings["busiest_stations_mapreduce_2"] = round(stations_s, 4)

print(f"===== Busiest stations — combined traffic (MapReduce Candidate 2) =====\nElapsed: {stations_s:.4f}s")
display(station_activity.head(10))

===== Busiest stations — combined traffic (MapReduce Candidate 2) =====
Elapsed: 0.0653s


,station_name,departures,arrivals,total_activity
0,Streeter Dr & Grand Ave,1916,1952,3868
1,Michigan Ave & Oak St,1151,1114,2265
2,Clark St & Elm St,1107,1083,2190
3,Wells St & Concord Ln,1063,1097,2160
4,Millennium Park,1047,1062,2109
5,Theater on the Lake,974,1007,1981
6,Wells St & Elm St,924,907,1831
7,Clark St & Lincoln Ave,894,847,1741
8,Indiana Ave & Roosevelt Rd,905,811,1716
9,Broadway & Barry Ave,805,901,1706


---
## Summary: Performance Baseline
All per-step timings, total wall time, and memory usage. Written to `metrics.json`.

In [8]:
notebook_end = time.perf_counter()
total_s = notebook_end - notebook_start

process = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / (1024 * 1024)

print("=" * 50)
print("PANDAS PROTOTYPE — TIMING SUMMARY")
print("=" * 50)
summary_rows = []
labels = {
    "load_preprocess"           : "Load & preprocess",
    "row_count_quality"         : "Row count & data quality",
    "rides_by_member_type"      : "Rides by member type",
    "rides_by_bike_type"        : "Rides by bike type",
    "avg_duration_by_member"    : "Avg duration by member type",
    "popular_routes_mapreduce_1": "Popular routes (MR Candidate 1)",
    "busiest_stations_mapreduce_2": "Busiest stations (MR Candidate 2)",
}
for key, label in labels.items():
    val = timings.get(key, 0)
    print(f"  {label:<38} {val:.4f}s")
    summary_rows.append({"step": label, "elapsed_s": val})

print("-" * 50)
print(f"  {'Total wall time':<38} {total_s:.4f}s")
print(f"  {'Memory (RSS)':<38} {memory_mb:.2f} MB")
print("=" * 50)

# Write to metrics.json
current_dir = os.getcwd()
results_dir = Path(current_dir) / "results"
if not results_dir.exists():
    results_dir = Path(current_dir).parent / "results"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_path = results_dir / "metrics.json"
existing = {}
if metrics_path.exists():
    try:
        existing = json.loads(metrics_path.read_text())
    except json.JSONDecodeError:
        pass

existing["pandas_prototype"] = {
    "steps"                       : timings,
    "total_execution_time_seconds": round(total_s, 2),
    "memory_usage_mb"             : round(memory_mb, 2),
}
metrics_path.write_text(json.dumps(existing, indent=4))
print(f"\nMetrics written to: {metrics_path.resolve()}")

PANDAS PROTOTYPE — TIMING SUMMARY
  Load & preprocess                      0.7700s
  Row count & data quality               0.0540s
  Rides by member type                   0.0297s
  Rides by bike type                     0.0231s
  Avg duration by member type            0.0635s
  Popular routes (MR Candidate 1)        0.1993s
  Busiest stations (MR Candidate 2)      0.0653s
--------------------------------------------------
  Total wall time                        1.4127s
  Memory (RSS)                           182.48 MB

Metrics written to: C:\Users\imonl\BigData\Cyclistic_Data\results\metrics.json
